# Cervecerías en Querétaro

Este notebook es un complemento a los mapas construidos (`maps/01_queretaro_cervecerias.png` y `maps/02_san_juan_del_rio_cervecerias.png`). En él utilizo Kepler.gl, una herramienta que facilita la construcción de mapas interactivos. Los archivos resultantes son `maps/05_brewings_qro.html` y `maps/06_voronoi.html`. 

Cada cervecera está localizada con un punto, donde el tamaño del círculo es el número de followers que cada cervecera tiene en su cuenta de Instagram (followers al día 3 de noviembre de 2026).

Nota: Una versión de esta herramienta la he utilizado en los últimos cuatro años. Sin embargo, es la primera vez que la uso en un notebook. El código fue construido utilizando Copilot. 

## Construccion del mapa y de un dataframe simplificado
Construí el dataframe simplificado `brewing_comp` que incluye únicamente las variables que utilizaré para la elaboración de los mapas. 

Adicional a la información del DENUE, incluí data externa recolectada de Instagram: el numero de followers (`ig_followers`), y las vistas de su video más viral (`views_viral_video`; extraido en la app de Instagram, no disponible para Cervecería Cuatro Palos). Como variable proxy de tamaño utilizaré `ig_followers` (asumo que el número de followers determina el alcance y popularidad de la marca). 

Nota: Por simplicidad, estoy tomando el número de followers, pero con la data recolectada se puede construir un cociente como `ig_followers`/`views_viral_video`. Este cociente puede ser un proxy del impacto del engagement de la empresa con las audiencias en redes sociales o el grado de posicionamiento de la marca.

In [1]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
from IPython.display import display
from keplergl import KeplerGl

root = Path.cwd()
data_dir = root / "data"
maps_dir = root / "maps"
geojson_dir = data_dir / "geojson"
maps_dir.mkdir(exist_ok=True)
geojson_dir.mkdir(exist_ok=True)

breweries = pd.read_csv(
    data_dir / "01_denue0526_ 312120_elab_cerveza.csv",
    encoding="utf-8-sig",
    dtype={"cve_mun": str},
)
with (data_dir / "brewing_comp_metrics.json").open(encoding="utf-8") as file:
    metrics = pd.DataFrame.from_dict(json.load(file), orient="index")
metrics.index.name = "nom_estab"
metrics = metrics.reset_index()

if set(metrics["nom_estab"]) != set(breweries["nom_estab"]):
    raise ValueError("JSON company names must match nom_estab values in the brewing CSV")
for column in ["ig_followers", "views_viral_video"]:
    metrics[column] = pd.to_numeric(metrics[column], errors="raise")

brewing_comp = breweries.merge(
    metrics, on="nom_estab", how="left", validate="one_to_one"
)
brewing_comp["norm_name"] = brewing_comp["nom_estab"].str.strip().str.title()
brewing_comp = brewing_comp[
    [
        "nom_estab",
        "per_ocu",
        "latitud",
        "longitud",
        "ig_followers",
        "views_viral_video",
        "norm_name",
    ]
]
brewing_comp

FileNotFoundError: [Errno 2] No such file or directory: '/Users/fernando.balandran/Desktop/fernando_balandran/data/brewing_comp_metrics.json'

In [2]:
municipalities = gpd.read_file(data_dir / "maps_polygons" / "22mun.shp")
municipalities["CVE_MUN"] = municipalities["CVE_MUN"].astype(str).str.zfill(3)
municipalities = municipalities.to_crs(epsg=4326)

brewing_points = gpd.GeoDataFrame(
    brewing_comp.assign(
        cve_mun=breweries["cve_mun"],
        map_label=brewing_comp["norm_name"]
        + "\nViews: "
        + brewing_comp["views_viral_video"].map(
            lambda views: f"{views:,.0f}" if pd.notna(views) else "N/D"
        ),
    ),
    geometry=gpd.points_from_xy(brewing_comp["longitud"], brewing_comp["latitud"]),
    crs="EPSG:4326",
)

kepler_maps = {}
for code, city in {"014": "queretaro", "016": "san_juan_del_rio"}.items():
    polygon = municipalities[municipalities["CVE_MUN"] == code]
    companies = brewing_points[brewing_points["cve_mun"] == code]
    if polygon.empty:
        raise ValueError(f"Municipality code {code} was not found in the shapefile")

    polygon.to_file(geojson_dir / f"{city}_poligono.geojson", driver="GeoJSON")
    companies.to_file(geojson_dir / f"{city}_cervecerias.geojson", driver="GeoJSON")

    kepler_config = {
        "version": "v1",
        "config": {
            "visState": {
                "filters": [],
                "layers": [
                    {
                        "id": "brewery-points",
                        "type": "point",
                        "config": {
                            "dataId": "Breweries",
                            "label": "Breweries",
                            "color": [31, 119, 180],
                            "columns": {"lat": "latitud", "lng": "longitud", "altitude": None},
                            "isVisible": True,
                            "visConfig": {
                                "radius": 10,
                                "fixedRadius": False,
                                "opacity": 0.8,
                                "outline": True,
                                "thickness": 2,
                                "radiusRange": [5, 40],
                                "filled": True,
                            },
                            "textLabel": [
                                {
                                    "field": {"name": "map_label", "type": "string"},
                                    "color": [0, 0, 0],
                                    "size": 14,
                                    "offset": [0, 0],
                                    "anchor": "start",
                                    "alignment": "center",
                                }
                            ],
                        },
                        "visualChannels": {
                            "colorField": None,
                            "colorScale": "quantile",
                            "sizeField": {"name": "ig_followers", "type": "integer"},
                            "sizeScale": "sqrt",
                        },
                    },
                    {
                        "id": "municipality-polygon",
                        "type": "geojson",
                        "config": {
                            "dataId": "Municipality",
                            "label": "Municipality",
                            "color": [220, 220, 220],
                            "columns": {"geojson": "_geojson"},
                            "isVisible": True,
                            "visConfig": {
                                "opacity": 0.18,
                                "thickness": 2,
                                "stroked": True,
                                "filled": True,
                                "color": [220, 220, 220],
                                "strokeColor": [40, 40, 40],
                            },
                        },
                        "visualChannels": {
                            "colorField": None,
                            "colorScale": "quantile",
                            "strokeColorField": None,
                            "strokeColorScale": "quantile",
                        },
                    },
                ],
                "interactionConfig": {
                    "tooltip": {
                        "fieldsToShow": {
                            "Breweries": [
                                {"name": "nom_estab", "format": None},
                                {"name": "ig_followers", "format": None},
                                {"name": "views_viral_video", "format": None},
                            ]
                        },
                        "compareMode": False,
                        "compareType": "absolute",
                    }
                },
                "layerBlending": "normal",
                "splitMaps": [],
                "animationConfig": {"currentTime": None, "speed": 1},
            },
            "mapStyle": {
                "styleType": "voyager",
                "visibleLayerGroups": {
                    "label": True,
                    "road": True,
                    "border": True,
                    "building": True,
                    "water": True,
                    "land": True,
                    "3d building": False,
                },
                "topLayerGroups": {"label": True},
            },
        },
    }


city_codes = ["014", "016"]
both_polygons = municipalities[municipalities["CVE_MUN"].isin(city_codes)]
both_companies = brewing_points[brewing_points["cve_mun"].isin(city_codes)]
if len(both_polygons) != len(city_codes):
    raise ValueError("Could not find both municipality polygons in the shapefile")

both_polygons.to_file(geojson_dir / "queretaro_san_juan_del_rio_poligonos.geojson", driver="GeoJSON")
both_companies.to_file(geojson_dir / "queretaro_san_juan_del_rio_cervecerias.geojson", driver="GeoJSON")
combined_map = KeplerGl(
    height=700,
    data={
        "Breweries": both_companies,
        "Municipality": json.loads(both_polygons.to_json()),
    },
    config=kepler_config,
    show_docs=False,
)
display(combined_map)

NameError: name 'brewing_comp' is not defined

In [8]:
output_path = maps_dir / "brewings_qro.html"
combined_map.save_to_html(
    file_name=str(output_path),
    data=combined_map.data,
    config=kepler_config,
    center_map=True,
)
print(f"Saved {output_path}")

Map saved to /Users/fernando.balandran/Desktop/fernando_balandran/maps/brewings_qro.html!
Saved /Users/fernando.balandran/Desktop/fernando_balandran/maps/brewings_qro.html


## Mapa de Voronoi de los comercios de cerveza

Este mapa Kepler usa el GeoJSON combinado con los polígonos de Voronoi de Querétaro y San Juan del Río. El color distingue el municipio.

In [9]:
voronoi_polygons = gpd.read_file(
    maps_dir / "queretaro_san_juan_del_rio_voronoi.geojson"
).to_crs(epsg=4326)
nse_polygons = gpd.read_file(data_dir / "nse_qro.geojson").to_crs(epsg=4326)
nse_ranks = {"A/B": 0, "C+": 1, "C": 2, "C-": 3, "D+": 4, "D": 5, "E": 6, "N/D": 7}
nse_polygons["nse_rank"] = nse_polygons["NIVEL"].map(nse_ranks)
if nse_polygons["nse_rank"].isna().any():
    raise ValueError("Unexpected NSE level in data/nse_qro.geojson")

voronoi_config = {
    "version": "v1",
    "config": {
        "visState": {
            "filters": [],
            "layers": [
                {
                    "id": "nse-polygons",
                    "type": "geojson",
                    "config": {
                        "dataId": "NSE",
                        "label": "NSE level",
                        "color": [178, 24, 43],
                        "columns": {"geojson": "_geojson"},
                        "isVisible": True,
                        "visConfig": {
                            "opacity": 0.75,
                            "thickness": 0,
                            "stroked": False,
                            "filled": True,
                            "colorRange": {
                                "name": "NSE A/B red to E white",
                                "type": "custom",
                                "category": "Custom",
                                "colors": [
                                    "#b2182b",
                                    "#d6604d",
                                    "#f4a582",
                                    "#fddbc7",
                                    "#f7f7f7",
                                    "#f0f0f0",
                                    "#ffffff",
                                    "#808080"
                                ]
                            }
                        }
                    },
                    "visualChannels": {
                        "colorField": {"name": "nse_rank", "type": "integer"},
                        "colorDomain": [0, 7],
                        "colorScale": "quantize",
                        "strokeColorField": None,
                        "strokeColorScale": "quantile"
                    }
                },
                {
                    "id": "voronoi-polygons",
                    "type": "geojson",
                    "config": {
                        "dataId": "Voronoi",
                        "label": "Beer commerce Voronoi",
                        "color": [0, 90, 255],
                        "columns": {"geojson": "_geojson"},
                        "isVisible": True,
                        "visConfig": {
                            "opacity": 1,
                            "thickness": 1,
                            "stroked": True,
                            "filled": False,
                            "color": [0, 0, 0],
                            "strokeColor": [0, 0, 0],
                        },
                    },
                    "visualChannels": {
                        "colorField": None,
                        "colorScale": "ordinal",
                        "strokeColorField": None,
                        "strokeColorScale": "quantile",
                    },
                }
            ],
            "interactionConfig": {
                "tooltip": {
                    "enabled": True,
                    "fieldsToShow": {
                        "Voronoi": [
                            {"name": "municipality", "format": None},
                            {"name": "nom_estab", "format": None},
                            {"name": "establishment_count", "format": None},
                        ],
                        "NSE": [
                            {"name": "NIVEL", "format": None},
                            {"name": "NOM_MUN", "format": None},
                            {"name": "NOM_LOC", "format": None}
                        ]
                    },
                    "compareMode": False,
                    "compareType": "absolute",
                }
            },
            "layerBlending": "normal",
            "splitMaps": [],
            "animationConfig": {"currentTime": None, "speed": 1},
        },
        "mapStyle": {
            "styleType": "voyager",
            "visibleLayerGroups": {
                "label": True,
                "road": True,
                "border": True,
                "building": True,
                "water": True,
                "land": True,
                "3d building": False,
            },
            "topLayerGroups": {
                "label": False,
                "road": False,
                "water": True,
                "land": True,
            },
        },
    },
}
voronoi_config["config"]["visState"]["layers"].reverse()

voronoi_map = KeplerGl(
    height=700,
    data={"NSE": nse_polygons, "Voronoi": voronoi_polygons},
    config=voronoi_config,
    show_docs=False,
)
display(voronoi_map)

In [ ]:
voronoi_output_path = maps_dir / "voronoi_qro.html"
voronoi_map.save_to_html(
    file_name=str(voronoi_output_path),
    data=voronoi_map.data,
    config=voronoi_config,
    center_map=True,
)
print(f"Saved {voronoi_output_path}")